# Data Cleansing

In [ ]:
# imports
import pandas as pd
import numpy as np

RANDOM_SEED=42

In [ ]:
df = pd.read_csv('./data/austin_car_crash.csv')
display(df.head())

# Drop rows

In [ ]:
# remove any rows that have 'Is temporary record` == True
print(df['Is temporary record'].value_counts())
print(len(df[df['Is temporary record'] == True]))
df = df[df['Is temporary record'] != True]
print(len(df[df['Is temporary record'] == True]))

In [ ]:
# remove any rows that have 'Is deleted' == True
print(df['Is deleted'].value_counts())
print(len(df[df['Is deleted'] == True]))
df = df[df['Is deleted'] != True]
print(len(df[df['Is deleted'] == True]))

# Drop columns

In [ ]:
print(df.columns)

In [ ]:
# Here I am removing columns that are not useful for the model
# IDs are not useful
# rpt columns are a repetition of Primary address
# No need for point as it was generated from the lat and long

to_drop = [ 'ID', 'Crash ID', 'case_id', 'rpt_block_num', 'rpt_street_name', 'rpt_street_sfx',
    'point', 'Is deleted', 'Is temporary record', 'Reported street prefix']
df.drop(to_drop, axis=1, inplace=True)

In [ ]:
display(df.head())

# Calculations

### Private or public roads

In [ ]:
# Combine onsys_fl and private_dr_fl into one column

print(df['onsys_fl'].value_counts())

In [ ]:
print(df['private_dr_fl'].value_counts())

There are no private accidents recorded.

In [ ]:
df.drop(['private_dr_fl', 'onsys_fl'], axis=1, inplace=True)

### Dates

In [ ]:
# count nan values for dates
print(df['Crash timestamp'].isna().sum())
print(df['Crash timestamp (US/Central)'].isna().sum())

In [ ]:
# Create a new column for better date and time handling
df.drop('Crash timestamp', axis=1, inplace=True)
df['Crash timestamp (US/Central)'] = pd.to_datetime(df['Crash timestamp (US/Central)'])

In [ ]:
print(df['Crash timestamp (US/Central)'].isna().sum())

In [ ]:
display(df['Crash timestamp (US/Central)'].head())

# Renaming columns

In [ ]:
print(df.columns)

In [ ]:
column_names = df.columns
new_column_names = {}
for column in column_names:
    new_column_names[column] = column.lower().replace(' ', '_')

print(new_column_names)

df.rename(columns=new_column_names, inplace=True)

In [ ]:
display(df.head())

In [ ]:
print(df.columns)

In [ ]:
renames = {
    'crash_fatal_fl': 'fatal_crash',
    'crash_speed_limit': 'speed_limit',
    'road_constr_zone_fl': 'construction_zone',
    'crash_sev_id': 'crash_severity',
    'crash_timestamp_(us/central)': 'timestamp_us_central',
}

In [ ]:
df.rename(columns=renames, inplace=True)
display(df.head())

# Value corrections

In [ ]:
df['speed_limit'].value_counts()

In [ ]:
df['speed_limit'] = df['speed_limit'].replace(-1, np.nan)
df['speed_limit'] = df['speed_limit'].replace(0, np.nan)

In [ ]:
df = df.dropna(subset=['speed_limit'])

In [ ]:
print(df['speed_limit'].max())

In [ ]:
print(df['speed_limit'].mean())

In [ ]:
# df['speed_limit'] = (np.ceil(df['speed_limit'] / 5) * 5).astype(int)
df['speed_limit'] = df['speed_limit'] // 5 * 5

In [ ]:
df['speed_limit'].value_counts()

In [ ]:
print(df['crash_severity'].value_counts())

In [ ]:
# Original: (0=UNKNOWN, 1=INCAPACITATING INJURY, 2=NON-INCAPACITATING INJURY, 3=POSSIBLE INJURY, 4=KILLED, 5=NOT INJURED)
# We want to change it to (0=NOT INJURED, 1=UNKNOWN,  2=POSSIBLE INJURY, 3=NON-INCAPACITATING INJURY, 4=INCAPACITATING INJURY, 5=KILLED)

severity_mapping = {
    0: 1,  # UNKNOWN ? becomes 1 (UNKNOWN)
    1: 4,  # INCAPACITATING INJURY ? becomes 4
    2: 3,  # NON-INCAPACITATING INJURY ? becomes 3
    3: 2,  # POSSIBLE INJURY ? becomes 2
    4: 5,  # KILLED ? becomes 5
    5: 0   # NOT INJURED ? becomes 0
}

# Apply the mapping
df['crash_severity'] = df['crash_severity'].replace(severity_mapping)

In [ ]:
print(df['crash_severity'].value_counts())

In [ ]:
severity_map = {
    0: "not_injured",
    1: "unknown",
    2: "possible_injury",
    3: "non_incapacitating_injury",
    4: "incapacitating_injury",
    5: "killed"
}

df["crash_severity"] = df["crash_severity"].map(severity_map)
df = pd.get_dummies(df, columns=["crash_severity"], prefix="severity")

# If we want to convert from True/False to 1/0
# cols_to_convert = [
#     "severity_not_injured",
#     "severity_unknown",
#     "severity_possible_injury",
#     "severity_non_incapacitating_injury",
#     "severity_incapacitating_injury",
#     "severity_killed"
# ]
# df[cols_to_convert] = df[cols_to_convert].astype(int)

display(df.head())

In [ ]:
print(df['construction_zone'].value_counts())

## Units involved

In [ ]:
print(df['units_involved'].value_counts())

In [ ]:
# Dropping small counts of units involved
value_counts = df['units_involved'].value_counts()
infrequent_values = value_counts[value_counts < 1000].index
df= df[~df['units_involved'].isin(infrequent_values)].copy()

In [ ]:
# Step 1: Get all unique entries from 'units_involved'
unique_entries = df['units_involved'].dropna().unique()

# Step 2: Split and flatten all individual vehicle types
vehicle_types = set()
for entry in unique_entries:
    types = [v.strip() for v in entry.split('&')]
    vehicle_types.update(types)

# remove spaces and convert to lowercase and remove any special characters
display(vehicle_types)

In [ ]:
# Sort for consistency
vehicle_types = sorted(vehicle_types)

# Create binary indicator columns for each vehicle type
for vehicle in vehicle_types:
    df[vehicle] = df['units_involved'].apply(
        lambda x: int(vehicle in x) if pd.notnull(x) else 0
    )

# Show the updated dataframe with the new binary columns
df[ ['units_involved']+vehicle_types]

In [ ]:
display(df.head())

In [ ]:
# rename the vehicle types to be more readable
to_rename = {}
for vehicle in vehicle_types:
    new_name = vehicle.replace('?', '').replace(' ', '_').replace('__', '_').replace('/', '_').replace('-', '_').lower()
    to_rename[vehicle] = 'unit_involved_' + new_name
print(to_rename)
df.rename(columns=to_rename, inplace=True)
display(df.head())
# remove the original units_involved column
df.drop('units_involved', axis=1, inplace=True)

## Timestamps

In [ ]:
from utils.utils import extract_timestamp

# Extract the timestamp features into a new DataFrame
timestamp_features = extract_timestamp(df["timestamp_us_central"])
display(timestamp_features)

# Concatenate the extracted features with the original DataFrame
df = pd.concat([df, timestamp_features], axis=1)
display(df.head())

In [ ]:
# Create cyclical features BEFORE splitting
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Remove Max Costs

In [ ]:
df.drop('estimated_maximum_comprehensive_cost', axis=1, inplace=True)

# Remove NAs

In [ ]:
display(df.isnull().sum())

In [ ]:
df = df.dropna()
display(df.isnull().sum())

# Save result

In [ ]:
df.to_csv('./data/austin_car_crash_cleaned.csv', index=False)

In [ ]:
df.shape